In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_teddynote import logging

logging.langsmith("CH12")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH12


In [3]:
import bs4
from langchain_classic import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma, FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

/Users/user/hanwha/rag_one/12/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/p6/pq5b_kqd3glc9ws5qzmfslkr0000gn/T/ipykernel_51707/2280978970.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
# 단계 1: 문서 로드(Load Documents)
# 뉴스기사 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
url = "https://n.news.naver.com/article/437/0000378416"
loader = WebBaseLoader(
    web_paths=(url,),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
        )
    ),
)
docs = loader.load()


# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

splits = text_splitter.split_documents(docs)

# 단계 3: 임베딩 & 벡터스토어 생성(Create Vectorstore)
# 벡터스토어를 생성합니다.
from langchain_huggingface import HuggingFaceEmbeddings
try:
    embeddings = OpenAIEmbeddings()

    # 실제 호출 테스트
    embeddings.embed_query("test")

    print("OpenAI 임베딩 사용")

except Exception as e:
    print(f"OpenAI 임베딩 사용 불가: {e}")

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "mps"},
        encode_kwargs={"normalize_embeddings": True},
    )

    print("HuggingFace 임베딩으로 대체")
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

# 단계 4: 검색(Search)
# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

# 단계 5: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
try:
    prompt = hub.pull("rlm/rag-prompt")
except Exception as e:
    from langsmith import Client

    client = Client()

    prompt = client.pull_prompt(
        "rlm/rag-prompt",
        dangerously_pull_public_prompt=True
    )

# 단계 6: 언어모델 생성(Create LLM)
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-6-luna")


def format_docs(docs):
    # 검색한 문서 결과를 하나의 문단으로 합쳐줍니다.
    return "\n\n".join(doc.page_content for doc in docs)


# 단계 7: 체인 생성(Create Chain)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "부영그룹의 출산 장려 정책에 대해 설명해주세요"
response = rag_chain.invoke(question)

# 결과 출력
print(f"URL: {url}")
print(f"문서의 수: {len(docs)}")
print("===" * 20)
print(f"[HUMAN]\n{question}\n")
print(f"[AI]\n{response}")

OpenAI 임베딩 사용 불가: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 54837.59it/s]


HuggingFace 임베딩으로 대체


/var/folders/p6/pq5b_kqd3glc9ws5qzmfslkr0000gn/T/ipykernel_51707/2548143761.py:51: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  prompt = hub.pull("rlm/rag-prompt")
python(51764) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


URL: https://n.news.naver.com/article/437/0000378416
문서의 수: 1
[HUMAN]
부영그룹의 출산 장려 정책에 대해 설명해주세요

[AI]
부영그룹은 2021년 이후 태어난 임직원 자녀에게 1명당 1억 원을 지급하고, 연년생이나 쌍둥이는 각각 지원해 총 2억 원을 받을 수 있도록 했습니다. 셋째까지 낳는 경우에는 국민주택을 제공하겠다고 밝혔으며, 이 정책을 앞으로도 이어갈 계획입니다.


In [5]:
# 뉴스기사의 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_paths=("https://www.bbc.com/news/business-68092814",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "main",
            attrs={"id": ["main-content"]},
        )
    ),
)
docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs[0].page_content[:500]

문서의 수: 1


''

In [6]:
from langchain_classic.document_loaders import PyPDFLoader

# PDF 파일 로드. 파일의 경로 입력
loader = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")

# 페이지 별 문서 로드
docs = loader.load()
print(f"문서의 수: {len(docs)}")

# 10번째 페이지의 내용 출력
print(f"\n[페이지내용]\n{docs[10].page_content[:500]}")
print(f"\n[metadata]\n{docs[10].metadata}\n")

문서의 수: 23

[페이지내용]
SPRi AI Brief |  
2023-12월호
8
코히어, 데이터 투명성 확보를 위한 데이터 출처 탐색기 공개
n 코히어와 12개 기관이  광범위한 데이터셋에 대한 감사를 통해 원본 데이터 출처, 재라이선스 상태, 
작성자 등 다양한 정보를 제공하는 ‘데이터 출처 탐색기’ 플랫폼을 출시
n 대화형 플랫폼을 통해 개발자는 데이터셋의 라이선스 상태를 쉽게 파악할 수 있으며 데이터셋의 
구성과 계보도 추적 가능
KEY Contents
£ 데이터 출처 탐색기, 광범위한 데이터셋 정보 제공을 통해 데이터 투명성 향상
n AI 기업 코히어(Cohere)가 매사추세츠 공과⼤(MIT), 하버드⼤ 로스쿨, 카네기멜론⼤ 등 12개 기관과 
함께 2023년 10월 25일 ‘데이터 출처 탐색기(Data Provenance Explorer)’ 플랫폼을 공개
∙ AI 모델 훈련에 사용되는 데이터셋의 불분명한 출처로 인해 데이터 투명성이 확보되지 않아 다양한 
법적·윤리적 문제가 발생
∙ 이에 연구진은 가

[metadata]
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': 'data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 10, 'page_label': '11'}



In [7]:
from langchain_community.document_loaders.csv_loader import CSVLoader

# CSV 파일 로드
loader = CSVLoader(file_path="data/titanic.csv")
docs = loader.load()
print(f"문서의 수: {len(docs)}")

# 10번째 페이지의 내용 출력
print(f"\n[페이지내용]\n{docs[10].page_content[:500]}")
print(f"\n[metadata]\n{docs[10].metadata}\n")

문서의 수: 891

[페이지내용]
PassengerId: 11
Survived: 1
Pclass: 3
Name: Sandstrom, Miss. Marguerite Rut
Sex: female
Age: 4
SibSp: 1
Parch: 1
Ticket: PP 9549
Fare: 16.7
Cabin: G6
Embarked: S

[metadata]
{'source': 'data/titanic.csv', 'row': 10}



In [8]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/appendix-keywords.txt")
docs = loader.load()
print(f"문서의 수: {len(docs)}")

# 10번째 페이지의 내용 출력
print(f"\n[페이지내용]\n{docs[0].page_content[:500]}")
print(f"\n[metadata]\n{docs[0].metadata}\n")

문서의 수: 1

[페이지내용]
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.
예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.
연관키워드: 토큰화, 자연어

[metadata]
{'source': 'data/appendix-keywords.txt'}



In [9]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(".", glob="data/*.txt", show_progress=True)
docs = loader.load()

print(f"문서의 수: {len(docs)}")

# 10번째 페이지의 내용 출력
print(f"\n[페이지내용]\n{docs[0].page_content[:500]}")
print(f"\n[metadata]\n{docs[0].metadata}\n")

  0%|          | 0/6 [00:00<?, ?it/s]python(51769) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
 17%|█▋        | 1/6 [00:01<00:09,  1.93s/it]libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
100%|██████████| 6/6 [00:02<00:00,  2.96it/s]

문서의 수: 6

[페이지내용]
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다. 예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다. 연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다. 예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다. 연관키워드: 자연어 처리, 벡터화, 딥러닝

Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다. 예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다. 연관키워드: 토큰화, 자연어

[metadata]
{'source': 'data/appendix-keywords-CP949.txt'}



In [10]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(".", glob="data/*.pdf")
docs = loader.load()

print(f"문서의 수: {len(docs)}\n")
print("[메타데이터]\n")
print(docs[0].metadata)
print("\n========= [앞부분] 미리보기 =========\n")
print(docs[0].page_content[2500:3000])

python(51795) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51796) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51815) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Matplotlib is building the font cache; this may take a moment.
No languages specified, defaulting to English.


문서의 수: 1

[메타데이터]

{'source': 'data/SPRI_AI_Brief_2023년12월호_F.pdf'}

========= [앞부분] 미리보기 =========

하는 기업에게 안전 테스트 결과와 시스템에 관한

주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및

AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진

△1026 플롭스(FLOPS, Floating Point Operation Per Second)를 초과하는 컴퓨팅 성능 또는 생물학적 서열 데이터를 주로 사용하고 1023플롭스를 초과하는 컴퓨팅 성능을 사용하는 모델 △단일 데이터센터에서 1,000Gbit/s 이상의 네트워킹으로 연결되며 AI 훈련에서 이론상 최대 1020 플롭스를 처리할 수 있는 컴퓨팅 용량을 갖춘 컴퓨팅 클러스터가 정보공유 요구대상

n (형평성과 시민권 향상) 법률, 주택, 보건 분야에서 AI의 무책임한 사용으로 인한 차별과 편견 및 기타

문제를 방지하는 조치를 확대

형사사법 시스템에서 AI 사용 모범사례를 개발하고, 주택 임대 시 AI 알고리즘 차별을 막기 위한 명확한

지침을 제공하며, 보건복지 


In [11]:
from langchain_community.document_loaders import PythonLoader

loader = DirectoryLoader(".", glob="**/*.py", loader_cls=PythonLoader)
docs = loader.load()

print(f"문서의 수: {len(docs)}\n")
print("[메타데이터]\n")
print(docs[0].metadata)
print("\n========= [앞부분] 미리보기 =========\n")
print(docs[0].page_content[:500])

문서의 수: 1

[메타데이터]

{'source': 'data/audio_utils.py'}

========= [앞부분] 미리보기 =========

import re
import os
from pytube import YouTube
from moviepy.editor import AudioFileClip, VideoFileClip
from pydub import AudioSegment
from pydub.silence import detect_nonsilent


def extract_abr(abr):
    youtube_audio_pattern = re.compile(r"\d+")
    kbps = youtube_audio_pattern.search(abr)
    if kbps:
        kbps = kbps.group()
        return int(kbps)
    else:
        return 0


def get_audio_filepath(filename):
    # audio 폴더가 없으면 생성
    if not os.path.isdir("audio"):
        os.mkdir("au


In [12]:
# 뉴스기사의 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_paths=("https://www.bbc.com/news/business-68092814",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "main",
            attrs={"id": ["main-content"]},
        )
    ),
)
docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs[0].page_content[:500]

문서의 수: 1


''

In [13]:
from langchain_classic.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=100,
    chunk_overlap=10,
    length_function=len,
    is_separator_regex=False,
)

In [26]:
# chain of density 논문의 일부 내용을 불러옵니다
with open("data/chain-of-density.txt", "r") as f:
    text = f.read()[:500]

text_splitter = CharacterTextSplitter(
    chunk_size=100, chunk_overlap=10, separator="\n\n"
)
print(len(text_splitter.split_text(text)))

text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=10, separator="\n")
print(len(text_splitter.split_text(text)))

text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=10, separator=" ")
print(len(text_splitter.split_text(text)))

text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0, separator=" ")
print(len(text_splitter.split_text(text)))

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100, separator=" ")
# text 파일을 청크로 나누어줍니다.
text_splitter.split_text(text)

1
2
6
6


['Selecting the “right” amount of information to include in a summary is a difficult task. \nA good summary should be detailed and entity-centric without being overly dense and hard to follow. To better understand this tradeoff, we solicit increasingly dense GPT-4 summaries with what we refer to as a “Chain of Density” (CoD) prompt. Specifically, GPT-4 generates an initial entity-sparse summary before iteratively incorporating missing salient entities without increasing the length. Summaries genera']

In [27]:
# 뉴스기사의 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_paths=("https://www.bbc.com/news/business-68092814",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "main",
            attrs={"id": ["main-content"]},
        )
    ),
)

# splitter 를 정의합니다.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100, separator=" ")

# 문서를 로드시 바로 분할까지 수행합니다.
split_docs = loader.load_and_split(text_splitter=text_splitter)
print(f"문서의 수: {len(docs)}")
docs[0].page_content[:500]

문서의 수: 1


''

In [28]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [29]:
recursive_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10,
    length_function=len,
    is_separator_regex=False,
)

In [30]:
with open("data/chain-of-density.txt", "r") as f:
    text = f.read()[:500]

In [33]:
character_text_splitter = CharacterTextSplitter(
    chunk_size=100, chunk_overlap=10, separator=" "
)
for sent in character_text_splitter.split_text(text):
    print(sent)
    print('-'*3*20)
print("="*3*20)
recursive_text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=100, chunk_overlap=10
)
for sent in recursive_text_splitter.split_text(text):
    print(sent)
    print('-'*3*20)

Selecting the “right” amount of information to include in a summary is a difficult task. 
A good
------------------------------------------------------------
A good summary should be detailed and entity-centric without being overly dense and hard to follow.
------------------------------------------------------------
to follow. To better understand this tradeoff, we solicit increasingly dense GPT-4 summaries with
------------------------------------------------------------
with what we refer to as a “Chain of Density” (CoD) prompt. Specifically, GPT-4 generates an initial
------------------------------------------------------------
an initial entity-sparse summary before iteratively incorporating missing salient entities without
------------------------------------------------------------
without increasing the length. Summaries genera
------------------------------------------------------------
Selecting the “right” amount of information to include in a summary is a difficult task.
--

In [34]:
recursive_text_splitter._separators

['\n\n', '\n', ' ', '']

In [35]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
try:
    embeddings = OpenAIEmbeddings()

    # 실제 호출 테스트
    embeddings.embed_query("test")

    print("OpenAI 임베딩 사용")

except Exception as e:
    print(f"OpenAI 임베딩 사용 불가: {e}")

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "mps"},
        encode_kwargs={"normalize_embeddings": True},
    )

    print("HuggingFace 임베딩으로 대체")

semantic_text_splitter = SemanticChunker(embeddings=embeddings, add_start_index=True)

OpenAI 임베딩 사용 불가: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 58062.41it/s]


HuggingFace 임베딩으로 대체


In [40]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open("data/chain-of-density.txt", "r") as f:
    text = f.read()[:500]

document = Document(
    page_content=text,
    metadata={"source": "data/chain-of-density.txt"},
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10,
)

splits = text_splitter.split_documents([document])

In [41]:
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings


try:
    embeddings = OpenAIEmbeddings()

    # 실제 호출 테스트
    embeddings.embed_query("test")

    print("OpenAI 임베딩 사용")

except Exception as e:
    print(f"OpenAI 임베딩 사용 불가: {e}")

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "mps"},
        encode_kwargs={"normalize_embeddings": True},
    )

    print("HuggingFace 임베딩으로 대체")

OpenAI 임베딩 사용 불가: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 38106.11it/s]


HuggingFace 임베딩으로 대체


In [42]:
vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)

In [44]:
query = "좋은 요약에 대해서 설명해줘."

retriever = vectorstore.as_retriever(search_type="similarity")
search_result = retriever.invoke(query)
print(search_result)

[Document(id='5c08059e-9b65-4f4f-8a87-261e637aae16', metadata={'source': 'data/chain-of-density.txt'}, page_content='A good summary should be detailed and entity-centric without being overly dense and hard to follow.'), Document(id='4c50162e-0005-4e26-b83e-9aee8d13b154', metadata={'source': 'data/chain-of-density.txt'}, page_content='Selecting the “right” amount of information to include in a summary is a difficult task.'), Document(id='356275f8-f579-4611-8d85-a435a2f3b25f', metadata={'source': 'data/chain-of-density.txt'}, page_content='without increasing the length. Summaries genera'), Document(id='b741ddeb-4a1e-49a2-aeb0-c63847dfb26f', metadata={'source': 'data/chain-of-density.txt'}, page_content='follow. To better understand this tradeoff, we solicit increasingly dense GPT-4 summaries with what')]


In [45]:
query = "CoD에 대해서 설명해줘."

retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 2})
search_result = retriever.invoke(query)
print(search_result)

[Document(id='3725614e-bb23-4298-8daa-8ecb1f5b4821', metadata={'source': 'data/chain-of-density.txt'}, page_content='with what we refer to as a “Chain of Density” (CoD) prompt. Specifically, GPT-4 generates an'), Document(id='5c08059e-9b65-4f4f-8a87-261e637aae16', metadata={'source': 'data/chain-of-density.txt'}, page_content='A good summary should be detailed and entity-centric without being overly dense and hard to follow.')]


In [56]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

query = "그.... LLM 활용 테크닉인데... C로 시작하고 D로 끝나는거... 그거 설명해줘."

llm = ChatOpenAI(model="gpt-6-luna")

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(), llm=llm
)

queries = retriever_from_llm.llm_chain.invoke({
    "question": query
})

print(queries)

['LLM 프롬프팅 기법 중 ‘C’로 시작하고 ‘D’로 끝나는 Chain of Density(CoD)가 무엇인지 설명해줘.', 'Chain of Density 기법은 요약문을 어떻게 점점 더 정보 밀도 높게 만드는지, LLM 활용 사례와 함께 알려줘.', 'LLM에서 CoD(Chain of Density) 프롬프팅을 사용하는 방법과 장단점을 설명해줘.']


In [52]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [53]:
unique_docs = retriever_from_llm.invoke(query)
len(unique_docs)

5

In [54]:
print(unique_docs)

[Document(id='3725614e-bb23-4298-8daa-8ecb1f5b4821', metadata={'source': 'data/chain-of-density.txt'}, page_content='with what we refer to as a “Chain of Density” (CoD) prompt. Specifically, GPT-4 generates an'), Document(id='b741ddeb-4a1e-49a2-aeb0-c63847dfb26f', metadata={'source': 'data/chain-of-density.txt'}, page_content='follow. To better understand this tradeoff, we solicit increasingly dense GPT-4 summaries with what'), Document(id='5c08059e-9b65-4f4f-8a87-261e637aae16', metadata={'source': 'data/chain-of-density.txt'}, page_content='A good summary should be detailed and entity-centric without being overly dense and hard to follow.'), Document(id='4c50162e-0005-4e26-b83e-9aee8d13b154', metadata={'source': 'data/chain-of-density.txt'}, page_content='Selecting the “right” amount of information to include in a summary is a difficult task.'), Document(id='356275f8-f579-4611-8d85-a435a2f3b25f', metadata={'source': 'data/chain-of-density.txt'}, page_content='without increasing the le

In [57]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

In [58]:
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings


try:
    embeddings = OpenAIEmbeddings()

    # 실제 호출 테스트
    embeddings.embed_query("test")

    print("OpenAI 임베딩 사용")

except Exception as e:
    print(f"OpenAI 임베딩 사용 불가: {e}")

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "mps"},
        encode_kwargs={"normalize_embeddings": True},
    )

    print("HuggingFace 임베딩으로 대체")

OpenAI 임베딩 사용 불가: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 54244.46it/s]


HuggingFace 임베딩으로 대체


In [59]:
doc_list = [
    "난 오늘 많이 먹어서 배가 정말 부르다",
    "떠나는 저 배가 오늘 마지막 배인가요?",
    "내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.",
]

bm25_retriever = BM25Retriever.from_texts(doc_list)
bm25_retriever.k = 2

faiss_vectorstore = FAISS.from_texts(doc_list, embedding=embeddings)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5]
)

In [60]:
def pretty_print(docs):
    for i, doc in enumerate(docs):
        print(f"[{i+1}] {doc.page_content}")

In [62]:
sample_query = "나 요즘 배에 정말 살이 많이 쪘어..."
print(f"[Query]\n{sample_query}\n")
relevant_docs = bm25_retriever.invoke(sample_query)
print("[BM25 Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = faiss_retriever.invoke(sample_query)
print("[FAISS Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = ensemble_retriever.invoke(sample_query)
print("[Ensemble Retriever]")
pretty_print(relevant_docs)

[Query]
나 요즘 배에 정말 살이 많이 쪘어...

[BM25 Retriever]
[1] 난 오늘 많이 먹어서 배가 정말 부르다
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[FAISS Retriever]
[1] 난 오늘 많이 먹어서 배가 정말 부르다
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[Ensemble Retriever]
[1] 난 오늘 많이 먹어서 배가 정말 부르다
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.


In [64]:
sample_query = "바다 위에 떠다니는 배들이 많다"
print(f"[Query]\n{sample_query}\n")
relevant_docs = bm25_retriever.invoke(sample_query)
print("[BM25 Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = faiss_retriever.invoke(sample_query)
print("[FAISS Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = ensemble_retriever.invoke(sample_query)
print("[Ensemble Retriever]")
pretty_print(relevant_docs)

[Query]
바다 위에 떠다니는 배들이 많다

[BM25 Retriever]
[1] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[FAISS Retriever]
[1] 난 오늘 많이 먹어서 배가 정말 부르다
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[Ensemble Retriever]
[1] 떠나는 저 배가 오늘 마지막 배인가요?
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[3] 난 오늘 많이 먹어서 배가 정말 부르다


In [66]:
sample_query = "ships"
print(f"[Query]\n{sample_query}\n")
relevant_docs = bm25_retriever.invoke(sample_query)
print("[BM25 Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = faiss_retriever.invoke(sample_query)
print("[FAISS Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = ensemble_retriever.invoke(sample_query)
print("[Ensemble Retriever]")
pretty_print(relevant_docs)

[Query]
ships

[BM25 Retriever]
[1] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[FAISS Retriever]
[1] 떠나는 저 배가 오늘 마지막 배인가요?
[2] 난 오늘 많이 먹어서 배가 정말 부르다
[Ensemble Retriever]
[1] 떠나는 저 배가 오늘 마지막 배인가요?
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[3] 난 오늘 많이 먹어서 배가 정말 부르다


In [67]:
sample_query = "pear"
print(f"[Query]\n{sample_query}\n")
relevant_docs = bm25_retriever.invoke(sample_query)
print("[BM25 Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = faiss_retriever.invoke(sample_query)
print("[FAISS Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = ensemble_retriever.invoke(sample_query)
print("[Ensemble Retriever]")
pretty_print(relevant_docs)

[Query]
pear

[BM25 Retriever]
[1] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[FAISS Retriever]
[1] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[2] 난 오늘 많이 먹어서 배가 정말 부르다
[Ensemble Retriever]
[1] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[3] 난 오늘 많이 먹어서 배가 정말 부르다


In [72]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True,
)

In [73]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-6-luna")

In [74]:
from langchain_classic.callbacks import get_openai_callback

with get_openai_callback() as cb:
    result = model.invoke("서울이 수도인 국가는 어디인가요?")
print(cb)

Tokens Used: 26
	Prompt Tokens: 16
		Prompt Tokens Cached: 0
	Completion Tokens: 10
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0


In [1]:
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings


try:
    embeddings = OpenAIEmbeddings()

    # 실제 호출 테스트
    embeddings.embed_query("test")

    print("OpenAI 임베딩 사용")

except Exception as e:
    print(f"OpenAI 임베딩 사용 불가: {e}")

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "mps"},
        encode_kwargs={"normalize_embeddings": True},
    )

    print("HuggingFace 임베딩으로 대체")

/var/folders/p6/pq5b_kqd3glc9ws5qzmfslkr0000gn/T/ipykernel_54309/2303244987.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


OpenAI 임베딩 사용 불가: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


/Users/user/hanwha/rag_one/12/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 53564.13it/s]


HuggingFace 임베딩으로 대체


In [ ]:
from langchain_classic.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

file_path = "data/SPRI_AI_Brief_2023년12월호_F.pdf"
loader = PyPDFLoader(file_path=file_path)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

split_docs = loader.load_and_split(text_splitter=text_splitter)

vectorstore = FAISS.from_documents(documents=split_docs, embedding=embeddings)


k=3

bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = k

faiss_vectorstore = FAISS.from_documents(split_docs, embedding=embeddings)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": k})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5]
)

from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True,
)

llm = ChatOpenAI(model_name="gpt-6-luna")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(f"PDF Path: {file_path}")
print(f"문서의 수: {len(split_docs)}")
print("="*3*20)

PDF Path: data/SPRI_AI_Brief_2023년12월호_F.pdf
문서의 수: 43


NameError: name 'question' is not defined

In [11]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "삼성 가우스에 대해 설명해주세요"

print(f"[HUMAN]\n{question}\n")

response = rag_chain.invoke(question)

print(response)

[HUMAN]
삼성 가우스에 대해 설명해주세요

삼성 가우스는 삼성전자가 개발해 2023년 공개한 생성형 AI 모델로, **언어·코드·이미지 모델**로 구성됩니다. 메일 작성·문서 요약·번역, 코드 생성, 이미지 생성·편집 등을 지원하며, 온디바이스 작동이 가능하도록 설계돼 사용자 정보가 외부로 유출될 위험을 줄이는 것이 특징입니다. 삼성전자는 이를 다양한 제품에 단계적으로 탑재할 계획이라고 밝혔습니다.


In [12]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "미래의 AI 소프트웨어 매출 전망은 어떻게 되나요?"
response = rag_chain.invoke(question)
print(response)

IDC는 AI 소프트웨어 매출이 2022년 640억 달러에서 2027년 2,510억 달러로 늘어 연평균 31.4% 성장할 것으로 전망합니다. 생성형 AI 플랫폼과 애플리케이션 매출도 2027년 283억 달러에 이를 것으로 예상됩니다.


In [ ]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "YouTube 가 2024년에 의무화 한 것은 무엇인가요?"
response = rag_chain.invoke(question)
print(response)

유튜브는 2024년부터 생성형 AI를 사용해 사실적으로 변경하거나 합성한 콘텐츠에 AI 라벨을 표시하도록 의무화했습니다. 표시하지 않으면 콘텐츠가 삭제되거나 수익 배분이 중단될 수 있습니다.


Failed to refresh cache entry rlm/rag-prompt: Connection error caused failure to GET /commits/rlm/rag-prompt/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/rlm/rag-prompt/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: lsv2_********************************************c0
Failed to refresh cache entry rlm/rag-prompt: Connection error caused failure to GET /commits/rlm/rag-prompt/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/rlm/rag-prompt/latest (Caused by NameResolutionError("HTTPSConnection(ho